# W2D3 — Cleaning a Messy Dataset — Guided

**Week 2 · Day 3 · Data Engineering & Preprocessing** · Lab

Today's dataset is hostile on purpose. It has missing values in three different patterns, dates in
four formats, duplicated rows, fourteen spellings of five cities, and one column that contains the
answer.

Every one of those was planted because the morning's session has a technique that fixes it, and a
technique you have only seen on a slide is not a technique you have.

The question that runs through the whole lab is never "which function do I call". It is **"why is
this value missing"** — because the answer changes what you are allowed to do about it. You will try
three imputation strategies on the same column and get three different scores, and then you will
find out that a leaking column had been making all three look equally good.

You'll leave with `cleaned.parquet` and `imputation_report.md`.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٢ اليوم ٣ — تنظيف بيانات فوضوية

**الأسبوع ٢ · اليوم ٣ · هندسة البيانات والمعالجة المسبقة** · معمل عملي

بيانات اليوم معطوبة عمدًا: فيها قيم مفقودة بثلاثة أنماط مختلفة، وتواريخ بأربع صيغ، وصفوف مكرّرة، وأربع
عشرة تهجئة لخمس مدن، وعمود واحد يحتوي الإجابة نفسها.

وكل عيب من هذه زُرع لأن في جلسة الصباح أسلوبًا يُصلحه، والأسلوب الذي رأيته في شريحة فقط ليس أسلوبًا
تملكه.

والسؤال الذي يسري في المعمل كله ليس «أي دالة أستدعي» بل **«لماذا هذه القيمة مفقودة»** — لأن الجواب
يغيّر ما يُسمح لك بفعله. وستجرّب ثلاث طرائق للتعويض على العمود نفسه فتحصل على ثلاث نتائج مختلفة، ثم
تكتشف أن عمودًا مُسرِّبًا كان يجعل الثلاث تبدو متساوية في الجودة.

ستخرج بملف `cleaned.parquet` وملف `imputation_report.md`.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Profile a table you have never seen and write down what is wrong with it before touching it.
- Remove duplicate rows and prove the removal did not change what you are predicting.
- Normalise inconsistent category spellings, and say why `value_counts()` is the tool that finds them.
- Parse a date column with more than one format, and check the result rather than trusting it.
- Choose between mean, constant and indicator-plus-mean imputation using the **cause** of the
  missingness, and report all three scores rather than only the one you chose.
- Recognise a leaking column from a correlation matrix and an exact arithmetic identity.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- فحص جدول لم تره من قبل وكتابة ما فيه من عيوب قبل أن تلمسه.
- حذف الصفوف المكرّرة وإثبات أن الحذف لم يغيّر ما تتنبّأ به.
- توحيد تهجئات الفئات غير المتّسقة، وبيان لماذا `value_counts()` هي الأداة التي تجدها.
- تحليل عمود تواريخ بأكثر من صيغة، والتحقّق من النتيجة بدل الوثوق بها.
- الاختيار بين التعويض بالمتوسط وبقيمة ثابتة وبمؤشّر مع المتوسط بناءً على **سبب** الفقدان، وعرض
  النتائج الثلاث كلها لا التي اخترتها فقط.
- التعرّف على عمود مُسرِّب من مصفوفة الارتباط ومن مطابقة حسابية تامة.

</div>

## About the data

**Dataset:** `messy_sales` — a classroom fixture, generated for this course · CC0 · 5,000 rows × 11 columns

Each row is **one sales order**. The columns are the order's identifier and date, the date it
shipped, the customer's city, the sales channel and the customer's tier, then the commercial numbers:
`quantity`, `unit_price`, `discount_pct`, `commission_paid`, and the target `revenue` — the amount
recognised on that order after the discount was applied and the invoice settled.

Predicting revenue from an order's own attributes is the shape of most forecasting work: you know
what was ordered and you want the amount before the money arrives.

The file was **built to be broken**, in five specific ways:

1. `discount_pct` is missing in **three different patterns**, and they demand different responses.
2. `order_date` is written in four formats, in one column.
3. There are exact duplicate rows.
4. `city` has fourteen spellings of five cities — case differences and stray whitespace.
5. One column should not be there at all.

**Watch out:** number 5 is the one that will cost you if you miss it. Nothing about it looks wrong.
It is a perfectly ordinary numeric column with a plausible name and no missing values, and a model
trained with it scores **R² = 1.0000** — which is not a triumph, it is the tell. **Task 6 is
finding it, and the task text does not name it.**

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `messy_sales` — بيانات صُنعت لهذه الدورة · ملكية عامة · ٥٬٠٠٠ صف × ١١ عمودًا

كل صف **طلب بيع واحد**. والأعمدة هي معرّف الطلب وتاريخه وتاريخ شحنه، ومدينة العميل، وقناة البيع وفئة
العميل، ثم الأرقام التجارية: الكمية وسعر الوحدة ونسبة الخصم والعمولة المدفوعة، والهدف `revenue` وهو
المبلغ المُعترف به على ذلك الطلب بعد تطبيق الخصم وتسوية الفاتورة.

والتنبّؤ بالإيراد من خصائص الطلب نفسه هو شكل معظم أعمال التنبّؤ: فأنت تعرف ما طُلب وتريد المبلغ قبل أن
يصل المال.

وقد **صُنع الملف ليكون معطوبًا** بخمس طرائق محدّدة:

١. العمود `discount_pct` مفقود بـ**ثلاثة أنماط مختلفة**، وكل نمط يقتضي تصرّفًا مختلفًا.
٢. العمود `order_date` مكتوب بأربع صيغ في عمود واحد.
٣. هناك صفوف مكرّرة تكرارًا تامًا.
٤. العمود `city` فيه أربع عشرة تهجئة لخمس مدن، باختلاف حالة الأحرف وبمسافات زائدة.
٥. وعمود واحد لا ينبغي أن يكون موجودًا أصلًا.

**انتبه:** العيب الخامس هو الذي يكلّفك إن أغفلته. ولا شيء في مظهره يبدو خطأً: فهو عمود رقمي عادي تمامًا
باسم معقول وبلا قيم مفقودة، والنموذج المُدرَّب معه يحقّق **R² يساوي ١٫٠٠٠٠** — وهذا ليس انتصارًا بل هو
العلامة. **والمهمة السادسة هي العثور عليه، ونصّ المهمة لا يسمّيه.**

</div>

## Setup

Run the cell below first. It also defines `score_frame`, which fits **the same model** — a plain
`LinearRegression` on an 80/20 split with a fixed seed — on whatever frame you hand it, and returns
the R² on the held-out fifth.

You are given that function so the lab stays about cleaning. Every score you compare today comes out
of it, which is what makes the comparisons fair: the only thing that changes between two numbers is
the cleaning decision you made.

<div dir="rtl" align="right">

## الإعداد

شغّل الخلية التالية أولًا. وهي تُعرّف أيضًا الدالة `score_frame` التي تُدرّب **النموذج نفسه** — انحدارًا
خطّيًا بسيطًا على تقسيم ٨٠٪ و٢٠٪ ببذرة ثابتة — على أي جدول تُعطيه إياها، وتُعيد قيمة R² على الخُمس
المحجوز.

وأُعطيت هذه الدالة ليبقى المعمل عن التنظيف. فكل نتيجة تقارنها اليوم تخرج منها، وهذا ما يجعل المقارنات
عادلة: فالشيء الوحيد الذي يتغيّر بين رقمين هو قرار التنظيف الذي اتّخذته.

</div>

In [ ]:
# === AIEP portable setup — works locally (Miniconda + uv) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    # A clone that never ran `uv pip install -e shared/` still has the package on disk —
    # use it before reaching for the network. Colab (no clone) falls through to pip.
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import get_dataset, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("scikit-learn", "matplotlib", "pyarrow")
seed_everything(42)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from aiep.viz import use_course_style
use_course_style()

DATA = get_dataset("messy_sales")

# The categorical columns get one-hot encoded; everything numeric goes in as-is.
CATEGORICAL = ["city", "channel", "customer_tier"]
TARGET = "revenue"


def score_frame(frame, numeric_cols):
    """Fit the same model on `frame` and return the R2 on a held-out 20%.

    A plain LinearRegression, an 80/20 split, random_state=42. Deliberately boring: the
    only thing that should differ between two calls is the cleaning you did to `frame`.
    """
    features = pd.get_dummies(
        frame[list(numeric_cols) + CATEGORICAL], columns=CATEGORICAL, drop_first=True
    ).astype("float64")
    y = frame[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        features, y, test_size=0.2, random_state=42
    )
    return LinearRegression().fit(X_train, y_train).score(X_test, y_test)


print(describe_dataset("messy_sales"))
print("\n", versions())

## Section 1 — Warm-up: profile it before you touch it  (≈25 min)

Everything in this section already works. Run it, read it, and **write down what is wrong before you
fix anything.** That order matters: once you start cleaning you lose the ability to see what the file
looked like, and "I fixed something" is not a record of what was broken.

Four questions, four one-line answers:

- What is missing? → `isna().sum()`
- How many distinct values does each column have? → `nunique()`
- Are there duplicate rows? → `duplicated().sum()`
- What do the categories actually look like? → `value_counts()`

<div dir="rtl" align="right">

## القسم الأول — التهيئة: افحصه قبل أن تلمسه (نحو ٢٥ دقيقة)

كل ما في هذا القسم يعمل أصلًا. شغّله واقرأه، و**اكتب ما فيه من عيوب قبل أن تُصلح شيئًا**. والترتيب مهم:
فبمجرّد أن تبدأ التنظيف تفقد القدرة على رؤية كيف كان الملف، وقولك «أصلحتُ شيئًا» ليس سجلًّا لما كان
معطوبًا.

أربعة أسئلة وأربع إجابات من سطر واحد:

- ما المفقود؟ ← `isna().sum()`
- كم قيمة مختلفة في كل عمود؟ ← `nunique()`
- هل توجد صفوف مكرّرة؟ ← `duplicated().sum()`
- كيف تبدو الفئات فعلًا؟ ← `value_counts()`

</div>

In [ ]:
raw = pd.read_csv(DATA)

print(f"rows: {len(raw):,}   columns: {raw.shape[1]}")
print(f"exact duplicate rows: {raw.duplicated().sum()}")
print()

profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(1),
    "distinct": raw.nunique(),
})
print(profile.to_string())

`city` reports five cities in the "About the data" section and something else here. `value_counts()`
shows you why: to pandas, `"Riyadh"`, `"riyadh"` and `"Riyadh "` are three unrelated values. A
trailing space is invisible in every output you have looked at so far.

This is the most common data defect in the world and it is entirely undramatic. It arrives whenever
two systems, or two people, typed into the same field.

<div dir="rtl" align="right">

يذكر قسم «عن البيانات» خمس مدن، ويظهر هنا رقم آخر. وتُظهر `value_counts()` السبب: فبالنسبة إلى pandas
تكون `"Riyadh"` و`"riyadh"` و`"Riyadh "` ثلاث قيم لا علاقة بينها. والمسافة في نهاية النص غير مرئية في
كل ما نظرت إليه من مخرجات حتى الآن.

وهذا أشيع عيب في البيانات في العالم، وهو غير مثير على الإطلاق. ويصل كلما كتب نظامان — أو شخصان — في
الحقل نفسه.

</div>

In [ ]:
print(f"distinct spellings of city: {raw['city'].nunique()}")
print()
print(raw["city"].value_counts().to_string())
print()
print("with the whitespace made visible:")
print([repr(v) for v in sorted(raw["city"].unique())])

And the date column. Print a sample and count how many shapes you can see.

<div dir="rtl" align="right">

ثم عمود التاريخ. اطبع عيّنة منه وعُدّ الصيغ التي تراها.

</div>

In [ ]:
print("a sample of order_date:")
print(raw["order_date"].head(12).tolist())
print()
print(f"missing values in discount_pct: {raw['discount_pct'].isna().sum():,} "
      f"({raw['discount_pct'].isna().mean():.1%})")
print("\nmissing rate of discount_pct, by channel:")
print(raw.groupby("channel")["discount_pct"].apply(lambda s: s.isna().mean())
         .to_string(float_format=lambda v: f"{v:.1%}"))

**Write down now, before Section 2:** the five things wrong with this file, and for the missing
values, your guess at *why* each pattern exists. The `by channel` breakdown above is a large clue for
one of the three patterns.

<div dir="rtl" align="right">

**اكتب الآن قبل القسم الثاني:** الأشياء الخمسة المعطوبة في هذا الملف، وبالنسبة للقيم المفقودة، خمّن
**لماذا** يوجد كل نمط. وتفصيل المفقود حسب القناة أعلاه دليل كبير على أحد الأنماط الثلاثة.

</div>

## Section 2 — Core: six repairs  (≈60 min)

1. Deduplicate — and prove the target distribution survived.
2. Normalise the city spellings.
3. Parse the dates, then check them.
4. Fit the model once. Believe nothing. **Find the column that should not be there.**
5. Three imputation strategies, three scores, one table.
6. Two encodings of a category column, and one of them invites a leak.

Task 4 comes before tasks 5 and 6 for a reason you will see the moment you run it: while a leaking
column is in the frame, **no preprocessing decision you make can be measured.** Every strategy scores
the same, because the model is reading the answer and nothing else can register. Finding the leak is
not the finale of this lab — it is the thing that has to happen before the rest of the lab means
anything.

Work on a copy called `clean` and leave `raw` untouched, so you can always look back at what the
file was.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ستّة إصلاحات (نحو ٦٠ دقيقة)

١. احذف المكرّر، وأثبت أن توزيع الهدف لم يتغيّر.
٢. وحّد تهجئات المدن.
٣. حلّل التواريخ ثم تحقّق منها.
٤. درّب النموذج مرة واحدة، ولا تصدّق شيئًا. **واعثر على العمود الذي لا ينبغي أن يكون موجودًا.**
٥. ثلاث طرائق تعويض، وثلاث نتائج، وجدول واحد.
٦. ترميزان لعمود فئوي، وأحدهما يفتح باب التسريب.

وتأتي المهمة الرابعة قبل الخامسة والسادسة لسبب ستراه لحظة تشغيلها: فما دام في الجدول عمود مُسرِّب،
**فلا يمكن قياس أي قرار معالجة تتّخذه.** إذ تحصل كل الطرائق على النتيجة نفسها، لأن النموذج يقرأ
الإجابة ولا يستطيع شيء آخر أن يظهر. فالعثور على التسريب ليس ختام هذا المعمل بل هو ما يجب أن يحدث قبل
أن يعني بقيّة المعمل شيئًا.

اعمل على نسخة اسمها `clean` واترك `raw` كما هو، لتستطيع دائمًا النظر إلى ما كان عليه الملف.

</div>

### Task 2.1 — deduplicate, and check what it cost

Dropping duplicates is one call. The part that is not one call is proving you did no damage.

A duplicate row is only safe to remove if it is a **recording** artefact — the same order written
twice — and not two genuinely separate orders that happen to match on every column. You usually
cannot tell for certain. What you can do is check that the thing you are predicting looks the same
before and after: if removing 120 rows moves the mean of `revenue` by 20%, those rows were not a
random sample of the file and you need to know why before you throw them away.

Report the mean and standard deviation of `revenue` before and after.

<div dir="rtl" align="right">

### المهمة ٢٫١ — احذف المكرّر، وتحقّق من ثمن ذلك

حذف المكرّر استدعاء واحد. أما ما ليس استدعاءً واحدًا فهو إثبات أنك لم تُحدث ضررًا.

فالصف المكرّر لا يكون حذفه آمنًا إلا إذا كان أثرًا **تسجيليًا** — الطلب نفسه مكتوبًا مرتين — لا طلبين
منفصلين حقًا تطابقا في كل عمود. وأنت لا تستطيع الجزم عادةً. لكن ما تستطيعه هو التحقّق أن ما تتنبّأ به
يبدو كما كان قبل الحذف وبعده: فإذا نقل حذف مئة وعشرين صفًا متوسط `revenue` بنسبة ٢٠٪ فتلك الصفوف لم
تكن عيّنة عشوائية من الملف، وعليك أن تعرف السبب قبل أن تتخلّص منها.

اذكر متوسط `revenue` وانحرافه المعياري قبل الحذف وبعده.

</div>

In [ ]:

before = {"rows": len(raw), "mean": raw[TARGET].mean(), "std": raw[TARGET].std()}

# TODO: Drop the exact duplicate rows into a new frame called `clean`, and reset the index.
# مهمة: احذف الصفوف المكرّرة تكرارًا تامًا في جدول جديد اسمه `clean`، وأعد تعيين الفهرس.
clean = ...

after = {"rows": len(clean), "mean": clean[TARGET].mean(), "std": clean[TARGET].std()}

# TODO: Print the before/after rows, mean and std, and the percentage change in the mean.
# مهمة: اطبع عدد الصفوف والمتوسط والانحراف قبل وبعد، ونسبة التغيّر في المتوسط.

120 rows went and the mean of `revenue` moved by about **0.1%**. That is what a safe deduplication
looks like: the rows you removed were an unremarkable sample of the file.

Keep the habit even when it is boring, because the interesting case looks identical up to this point.
Duplicates that cluster — every order from one branch imported twice — shift the target and change
which model wins, and the only thing that tells you is this comparison.

<div dir="rtl" align="right">

حُذف مئة وعشرون صفًا وتغيّر متوسط `revenue` بنحو **٠٫١٪**. وهذا شكل الحذف الآمن للمكرّر: فالصفوف
المحذوفة كانت عيّنة غير مميّزة من الملف.

واحفظ هذه العادة حتى إذا كانت مملّة، لأن الحالة المثيرة تبدو مطابقة لهذه إلى هذه النقطة. فالمكرّرات
المتكدّسة — كأن تُستورد طلبات فرع واحد مرتين — تُزيح الهدف وتغيّر أي نموذج يفوز، ولا يخبرك بذلك إلا هذه
المقارنة.

</div>

### Task 2.2 — one city, one spelling

Strip the whitespace, unify the case, and count again. Fourteen should become five.

Two warnings that generalise past this file:

- **Do this before anything that groups.** A `groupby("city")` on the raw column reports fourteen
  cities with wrong sizes, and nothing in the output looks broken.
- **`.str.title()` is not a general solution.** It is right for `riyadh` and wrong for `mcdonald` and
  for most real place names with internal punctuation. For a real project you build a mapping table
  and review it by hand. Five cities is small enough that the shortcut is honest here — say so rather
  than believing that lowercase-then-title is a cleaning strategy.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — مدينة واحدة وتهجئة واحدة

احذف المسافات الزائدة، ووحّد حالة الأحرف، ثم أعد العدّ. فينبغي أن تصبح الأربع عشرة خمسًا.

وتحذيران يتجاوزان هذا الملف:

- **افعل هذا قبل أي شيء يُجمّع.** فتطبيق `groupby("city")` على العمود الخام يُبلّغ عن أربع عشرة مدينة
  بأحجام خاطئة، ولا شيء في المخرج يبدو معطوبًا.
- **الدالة `.str.title()` ليست حلًّا عامًا.** فهي صحيحة مع `riyadh` وخاطئة مع `mcdonald` ومع معظم أسماء
  الأماكن الحقيقية التي فيها علامات داخلية. وفي مشروع حقيقي تبني جدول مطابقة وتراجعه يدويًا. وخمس مدن
  عدد صغير يجعل الاختصار صادقًا هنا — فصرّح بذلك بدل أن تعتقد أن التصغير ثم الترويس أسلوب تنظيف.

</div>

In [ ]:

spellings_before = clean["city"].nunique()

# TODO: Normalise the city column so each city has exactly one spelling.
# مهمة: وحّد عمود المدينة بحيث يكون لكل مدينة تهجئة واحدة بالضبط.

print(f"distinct spellings: {spellings_before} -> {clean['city'].nunique()}")
print()
print(clean["city"].value_counts().to_string())

### Task 2.3 — four date formats in one column

`order_date` contains `2025-03-04`, `03/26/2024`, `8 Dec 2024` and `2024/11/30`. One column, four
shapes, and no marker telling you which row is which.

`pd.to_datetime(..., format="mixed")` handles that: it infers the format **per row** rather than
demanding one for the whole column. It is allowed here. What is not allowed is calling it and moving
on — inference can be wrong, and when it is wrong it is silent. So:

- **check nothing came back as `NaT`.** A failed parse is a missing date, and a missing date will
  quietly drop rows out of any time-based analysis.
- **check no date is in the future.** This is what catches a day/month swap: parsing `13/07/2024` as
  month 13 fails loudly, but `07/13/2024` read the wrong way round gives you a date that is real and
  wrong, and future dates are how you notice.

Also parse `ship_date`, and confirm nothing shipped before it was ordered. Same idea: a check that
would catch the swap.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — أربع صيغ تاريخ في عمود واحد

يحتوي `order_date` على `2025-03-04` و`03/26/2024` و`8 Dec 2024` و`2024/11/30`. عمود واحد بأربع صيغ،
وبلا علامة تخبرك أي صف بأي صيغة.

وتتعامل `pd.to_datetime(..., format="mixed")` مع هذا: فهي تستنتج الصيغة **لكل صف** بدل أن تطلب صيغة
واحدة للعمود كله. وهي مسموحة هنا. أما غير المسموح فهو استدعاؤها والمتابعة، لأن الاستنتاج قد يكون خاطئًا
وهو خاطئ بصمت. إذًا:

- **تحقّق أن شيئًا لم يعد بقيمة `NaT`.** فالتحليل الفاشل تاريخ مفقود، والتاريخ المفقود يُسقط صفوفًا
  بهدوء من أي تحليل زمني.
- **تحقّق أن لا تاريخ في المستقبل.** وهذا ما يكشف تبديل اليوم بالشهر: فتحليل `13/07/2024` على أن ١٣
  شهر يفشل بصوت مسموع، أما `07/13/2024` مقروءًا معكوسًا فيعطيك تاريخًا حقيقيًا وخاطئًا، والتواريخ
  المستقبلية هي كيف تلاحظ ذلك.

وحلّل أيضًا `ship_date`، وتأكّد أن شيئًا لم يُشحن قبل أن يُطلب. الفكرة نفسها: تحقّق يكشف التبديل.

</div>

In [ ]:

# TODO: Parse order_date and ship_date, letting the format vary per row.
# مهمة: حلّل order_date وship_date مع السماح للصيغة بالتغيّر لكل صف.

# TODO: Count the failed parses, the future dates, and the negative shipping delays.
# مهمة: احسب التحليلات الفاشلة والتواريخ المستقبلية وأزمنة الشحن السالبة.

print(f"failed to parse:            {n_unparsed}")
print(f"dates in the future:        {n_future}")
print(f"shipped before ordered:     {n_before_order}")
print(f"\ndate range: {clean['order_date'].min().date()} "
      f"to {clean['order_date'].max().date()}")

### Task 2.4 — fit it once, and refuse to be pleased

The frame is now tidy enough to model. Fit it — filling the missing discounts with the median for
now, so that something can be fitted at all — and read the R².

Then stop, and read it again.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — درّبه مرة واحدة، وارفض أن تكون مسرورًا

صار الجدول مرتّبًا بما يكفي للنمذجة. فدرّبه — مع ملء الخصومات المفقودة بالوسيط مؤقتًا ليصبح التدريب
ممكنًا — واقرأ قيمة R².

ثم توقّف واقرأها مرة أخرى.

</div>

In [ ]:

ALL_NUMERIC = ["quantity", "unit_price", "discount_pct", "commission_paid"]

# TODO: Fill the missing discounts with the median and score the frame as it stands.
# مهمة: املأ الخصومات المفقودة بالوسيط وقيّم الجدول كما هو.

print(f"R2 on held-out data: {first_r2:.4f}")

**R² = 1.0000.**

That is not a good result. It is not a result at all. A perfect score on held-out data means the
model is not predicting `revenue` — it is **reading** it. Somewhere in those columns the answer is
present, and the fit found it in one line.

Treat this the way you would treat a test suite that passes before you wrote the code. The reflex to
build is: **an implausibly good score is a bug report about your features.** Not a triumph, not
something to screenshot, and above all not something to report before you have found out why.

So find out why. Three steps, and they are the ones that generalise to a dataset nobody warned you
about:

1. **Correlate every numeric column against the target and sort.** A feature correlating at 1.000
   with what you are predicting is either the discovery of the century or a mistake.
2. **Test for an exact identity.** A high correlation can be legitimate. Divide the suspect by the
   target and look at the result — if it is the same number on all 4,880 rows, the column is a
   restatement of the answer and not a measurement of anything.
3. **Ask when the value becomes known.** This is the step that decides it, and no amount of staring
   at correlations substitutes for it: *could you fill this column in for an order placed one minute
   ago?* If no, the column cannot be in the model, however predictive it is.

<div dir="rtl" align="right">

**R² يساوي ١٫٠٠٠٠.**

وهذه ليست نتيجة جيّدة، بل ليست نتيجة أصلًا. فالنتيجة الكاملة على بيانات محجوزة تعني أن النموذج لا
يتنبّأ بـ `revenue` بل **يقرأه**. فالإجابة موجودة في مكان ما بين تلك الأعمدة، ووجدها التدريب في سطر
واحد.

وتعامل مع هذا كما تتعامل مع مجموعة اختبارات تنجح قبل أن تكتب الشيفرة. والعادة التي تبنيها هي:
**النتيجة الجيّدة على نحو غير معقول تقرير عن عيب في خصائصك.** لا انتصار، ولا شيء يُصوَّر، وقبل كل شيء
لا شيء يُعرض قبل أن تعرف السبب.

فاعرف السبب. ثلاث خطوات، وهي التي تصلح لبيانات لم يحذّرك منها أحد:

١. **احسب ارتباط كل عمود رقمي بالهدف ورتّب.** فالخاصية التي ترتبط بما تتنبّأ به بمعامل ١٫٠٠٠ إمّا كشف
   القرن أو خطأ.
٢. **افحص وجود مطابقة تامة.** فالارتباط العالي قد يكون مشروعًا. اقسم المتّهم على الهدف وانظر في
   الناتج: فإن كان الرقم نفسه في كل الصفوف الأربعة آلاف وثمانمئة وثمانين فالعمود إعادة صياغة للإجابة
   لا قياس لشيء.
٣. **اسأل متى تُعرف القيمة.** وهذه هي الخطوة التي تحسم الأمر، ولا يُغني عنها أي قدر من التأمّل في
   الارتباطات: *هل تستطيع ملء هذا العمود لطلب قُدِّم قبل دقيقة؟* فإن كان الجواب لا فلا يجوز أن يكون
   العمود في النموذج مهما كانت قدرته التنبؤية.

</div>

In [ ]:

numeric_columns = clean.select_dtypes("number").columns.drop(["order_id", TARGET])

# TODO: Correlate every numeric column with the target, strongest first.
# مهمة: احسب ارتباط كل عمود رقمي بالهدف، والأقوى أولًا.

# TODO: Divide the suspect by the target and describe the ratio.
# مهمة: اقسم المتّهم على الهدف وصِف النسبة الناتجة.

`commission_paid` is **3% of `revenue`**, rounded to the cent, on every row.

Now step 3, which is the one that decides it. A sales commission is paid **after** the revenue is
recognised. For an order placed a minute ago there is no commission — the field does not exist yet,
and nothing you know at order time produces it: it is not derivable from `quantity`, `unit_price`,
`discount_pct`, the city, the channel or the tier. It is the answer, arriving from the future, scaled
by 0.03.

That distinction is worth being precise about, because the *other* kind of redundant column is not a
leak. If this file had contained an `invoice_total` equal to `unit_price * quantity`, that would be a
duplicated calculation — annoying for a linear model's coefficients, but perfectly legitimate,
because you can compute it yourself the moment the order is placed. The test is not "is this column
suspiciously predictive". The test is **"when does this value come into existence"**, and only the
commission fails it.

Drop it, and see what your cleaning was actually worth.

<div dir="rtl" align="right">

العمود `commission_paid` هو **٣٪ من `revenue`** مقرّبًا إلى القرش، في كل صف.

والآن الخطوة الثالثة وهي الحاسمة. فعمولة البيع تُدفع **بعد** الاعتراف بالإيراد. ولطلب قُدِّم قبل دقيقة
لا توجد عمولة: الحقل غير موجود بعد، ولا شيء تعرفه وقت الطلب يُنتجه — فهو غير مُستخرَج من الكمية ولا من
سعر الوحدة ولا من نسبة الخصم ولا من المدينة ولا القناة ولا الفئة. إنه الإجابة قادمةً من المستقبل
مضروبةً في ٠٫٠٣.

ويستحق هذا التمييز دقّة، لأن النوع **الآخر** من الأعمدة المكرّرة ليس تسريبًا. فلو كان في هذا الملف عمود
`invoice_total` يساوي حاصل `unit_price * quantity` لكان حسابًا مكرّرًا — مُزعجًا لمعاملات النموذج
الخطّي لكنه مشروع تمامًا، لأنك تستطيع حسابه بنفسك لحظة تقديم الطلب. فالفحص ليس «هل هذا العمود قوي
التنبّؤ على نحو مريب»، بل الفحص هو **«متى يوجد هذا الرقم»**، والعمولة وحدها هي التي تفشل فيه.

احذفه وانظر ما كانت قيمة تنظيفك فعلًا.

</div>

In [ ]:

LEAK = "commission_paid"
HONEST_NUMERIC = ["quantity", "unit_price", "discount_pct"]

# TODO: Drop the leaking column from `clean`.
# مهمة: احذف العمود المُسرِّب من `clean`.
clean = ...

# TODO: Re-score without it, using the same model and the same split.
# مهمة: أعد التقييم بدونه، بالنموذج نفسه والتقسيم نفسه.

print(f"with the leak:    R2 = {first_r2:.4f}")
print(f"without it:       R2 = {honest_r2:.4f}")
print(f"the difference:        {honest_r2 - first_r2:+.4f}")

**1.0000 becomes 0.7885.** A fifth of the variance you thought you were explaining was a column
being copied back to you.

0.79 is the number that goes in your report. Had you shipped the 1.0, you would have been asked —
correctly — why the live system performs so much worse than the notebook, and the answer would have
been that it never performed that well at all.

Notice how ordinary the column looked: a sensible name, no missing values, plausible magnitudes, and
nothing in any output flagging it. This is why leakage survives code review, and why W2D5 spends a
whole session on the version that is far harder to see than this one.

**Now the rest of the lab can be measured.** Everything below this point moves the score by
thousandths rather than by fifths, which is what real preprocessing decisions look like once the
answer is not in the features.

<div dir="rtl" align="right">

**تحوّلت ١٫٠٠٠٠ إلى ٠٫٧٨٨٥.** فخُمس التفاوت الذي حسبت أنك تُفسّره كان عمودًا يُعاد نسخه إليك.

والرقم ٠٫٧٩ هو ما يوضع في تقريرك. ولو سلّمت الرقم ١٫٠ لسُئلت — بحق — لماذا يعمل النظام الحقيقي أسوأ
بكثير من الدفتر، ولكان الجواب أنه لم يعمل بتلك الجودة قط.

ولاحظ كم بدا العمود عاديًا: اسم معقول، وبلا قيم مفقودة، وبمقادير مقبولة، ولا شيء في أي مخرج يُشير
إليه. ولهذا ينجو التسريب من مراجعة الشيفرة، ولهذا يُخصّص اليوم الخامس جلسة كاملة للصيغة الأصعب رؤيةً
من هذه.

**والآن يمكن قياس بقيّة المعمل.** فكل ما بعد هذه النقطة يُحرّك النتيجة بأجزاء الألف لا بالأخماس، وهذا
هو شكل قرارات المعالجة الحقيقية بعد أن تخرج الإجابة من الخصائص.

</div>

### Task 2.5 — three imputations, three scores

`discount_pct` is missing for roughly **29%** of orders, in three patterns. You saw one of them in
the warm-up: **every phone order is missing**, because that channel never had the field.

Three strategies, each on **its own copy** of the frame so they do not contaminate each other:

| Strategy | What it assumes |
|---|---|
| **median** | the missing discounts look like the ones you can see |
| **constant 0** | a missing discount means no discount was given |
| **indicator + median** | the *fact* that it is missing is itself information, so keep it as a column |

Fit the same model on all three. Report all three numbers in a table, including the one you will not
choose — a strategy you rejected with a measurement beside it is a decision; a strategy you rejected
without one is a preference.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — ثلاث طرائق تعويض وثلاث نتائج

العمود `discount_pct` مفقود في نحو **٢٩٪** من الطلبات بثلاثة أنماط، ورأيت أحدها في التهيئة: **كل طلبات
الهاتف مفقودة**، لأن تلك القناة لم يكن فيها هذا الحقل أصلًا.

ثلاث طرائق، كل واحدة على **نسختها** من الجدول كي لا تُلوّث إحداها الأخرى:

| الطريقة | ما تفترضه |
|---|---|
| **الوسيط** | أن الخصومات المفقودة تشبه التي تراها |
| **قيمة ثابتة صفر** | أن الخصم المفقود يعني أنه لم يُمنح خصم |
| **مؤشّر مع الوسيط** | أن **كون** القيمة مفقودة معلومة بنفسها، فتُحفظ عمودًا |

درّب النموذج نفسه على الثلاث. واذكر الأرقام الثلاثة في جدول، ومنها الرقم الذي لن تختاره — فالطريقة التي
ترفضها ومعها قياس قرار، والطريقة التي ترفضها بلا قياس تفضيل.

</div>

In [ ]:

# TODO: Copy A — fill the missing discounts with the column median.
# مهمة: النسخة الأولى — املأ الخصومات المفقودة بوسيط العمود.

# TODO: Copy B — fill them with the constant 0.
# مهمة: النسخة الثانية — املأها بالقيمة الثابتة صفر.

# TODO: Copy C — record where the value was missing in a new column, then fill the median.
# مهمة: النسخة الثالثة — سجّل موضع الفقدان في عمود جديد، ثم املأ بالوسيط.

# TODO: Score all three with the same model, and collect the results into a table.
# مهمة: قيّم الثلاث بالنموذج نفسه، واجمع النتائج في جدول.

print(imputation_scores.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nrow count is identical across all three: "
      f"{len(by_median) == len(by_zero) == len(by_indicator) == len(clean)}")

The indicator wins, the median is second, the constant 0 is last — and the whole spread is about
**0.009 of R²**. Small, and in the direction the *cause* predicted: phone orders are not
zero-discount orders, they are orders whose discount was never recorded, so telling the model where
the gap was beats guessing at what filled it.

Note carefully what you have and have not shown. Three scores from one split, differing in the third
decimal, is weak evidence on its own — a different `random_state` could reorder the bottom two. The
reason to prefer the indicator is not that it won by 0.003. It is that the mechanism is right, and
the measurement did not contradict it. W2D5 gives you the tool that turns this into real evidence:
five splits instead of one, and a spread you can compare against.

<div dir="rtl" align="right">

فاز المؤشّر، وجاء الوسيط ثانيًا، والقيمة الثابتة صفر أخيرًا — والفرق كله نحو **٠٫٠٠٩ من R²**. وهو صغير،
وفي الاتجاه الذي تنبّأ به **السبب**: فطلبات الهاتف ليست طلبات بخصم صفر، بل طلبات لم يُسجَّل خصمها قط،
فإخبار النموذج بموضع الفراغ أفضل من تخمين ما يملؤه.

ولاحظ بدقّة ما أثبتته وما لم تُثبته. فثلاث نتائج من تقسيم واحد تختلف في المنزلة العشرية الثالثة دليل
ضعيف بنفسه — وقد تُعيد بذرة عشوائية أخرى ترتيب الأخيرتين. وسبب تفضيل المؤشّر ليس أنه فاز بـ ٠٫٠٠٣، بل
أن الآليّة صحيحة وأن القياس لم يخالفها. ويعطيك اليوم الخامس الأداة التي تحوّل هذا إلى دليل حقيقي: خمسة
تقسيمات بدل واحد، ومدى تفاوت تقارن به.

</div>

### Task 2.6 — one-hot against target encoding

Two ways to turn `city` into numbers:

- **One-hot** — one 0/1 column per city. No ordering implied, no information about the target used.
  Costs you a column per category, which matters when there are four thousand of them and not five.
- **Target encoding** — replace each city with the mean `revenue` of that city. One column instead of
  five, and it carries real signal.

Target encoding has a trap, and it is the reason this task exists. Those group means are computed
**from the target**. If you compute them over the whole dataset and then split, every training row
has been told something about the test rows, and your score goes up for a reason that will not exist
in production.

So: compute the means **on the training rows only**, and map them onto both halves. Then, to see the
size of the trap, compute them over everything and score that too.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — الترميز الأحادي مقابل الترميز بالهدف

طريقتان لتحويل `city` إلى أرقام:

- **الترميز الأحادي (one-hot)** — عمود بقيم ٠ و١ لكل مدينة. لا يفرض ترتيبًا ولا يستخدم أي معلومة عن
  الهدف. ويكلّفك عمودًا لكل فئة، وهذا يهمّ حين تكون الفئات أربعة آلاف لا خمسًا.
- **الترميز بالهدف (target encoding)** — استبدال كل مدينة بمتوسط `revenue` فيها. عمود واحد بدل خمسة،
  ويحمل إشارة حقيقية.

وللترميز بالهدف فخّ، وهو سبب وجود هذه المهمة: فمتوسطات المجموعات تُحسب **من الهدف**. فإذا حسبتها على
البيانات كلها ثم قسّمت، صار كل صف تدريب قد أُخبر بشيء عن صفوف الاختبار، وارتفعت نتيجتك لسبب لن يوجد في
الإنتاج.

إذًا: احسب المتوسطات **على صفوف التدريب فقط** وطبّقها على النصفين. ثم لترى حجم الفخّ، احسبها على كل
الصفوف وقيّم ذلك أيضًا.

</div>

In [ ]:

frame = by_indicator
NUMERIC = HONEST_NUMERIC + ["discount_was_missing"]

# TODO: Score the one-hot version.
# مهمة: قيّم صيغة الترميز الأحادي.
onehot_r2 = ...


def target_encode_and_score(frame, numeric_cols, train_only):
    """Replace `city` with its mean revenue and score. Means from train rows, or from all."""
    # TODO: means the same thing in both places.
    # مهمة: «التدريب فقط» واحدًا في الموضعين.


# TODO: Score target encoding fitted on train only, then fitted on everything.
# مهمة: قيّم الترميز بالهدف محسوبًا على التدريب فقط، ثم محسوبًا على كل الصفوف.

print(f"one-hot                        R2 = {onehot_r2:.4f}")
print(f"target encoding, train only    R2 = {target_train_r2:.4f}")
print(f"target encoding, all rows      R2 = {target_all_r2:.4f}   <- leaked")
print(f"\nwhat the leak bought:          {target_all_r2 - target_train_r2:+.4f}")

The three land within **0.0005** of each other, and the leaked version is ahead by 0.0002.

Read that carefully, because there are two wrong conclusions available and one right one.

- ❌ *"Target encoding beat one-hot."* By 0.0004, on one split. That is not a result, it is noise
  wearing a decimal point. The honest statement is that this column is too weak for the choice of
  encoding to matter.
- ❌ *"Target encoding is safe, then."* No. The leak is worth nothing **here**, and the reason is
  arithmetic: with five cities, each group mean is computed from roughly a thousand rows and barely
  knows about any individual one. With 4,000 cities each mean is computed from one or two rows and
  is essentially the target written into the feature. Same code, catastrophic difference. The damage
  scales with **cardinality**.
- ✅ *"On this column, the encoding choice does not matter, and I know why."* Which also tells you
  where to spend your attention instead — the missingness, which was worth 0.009, and the leak, which
  was worth 0.21.

<div dir="rtl" align="right">

النتائج الثلاث في حدود **٠٫٠٠٠٥** من بعضها، والنسخة المُسرِّبة أعلى بـ ٠٫٠٠٠٢.

واقرأ ذلك بتمعّن، فهناك استنتاجان خاطئان متاحان واستنتاج صحيح واحد.

- ❌ *«تفوّق الترميز بالهدف على الأحادي.»* بفارق ٠٫٠٠٠٤ على تقسيم واحد. وهذه ليست نتيجة بل ضوضاء تلبس
  فاصلة عشرية. والعبارة الصادقة أن هذا العمود أضعف من أن يهمّ فيه اختيار الترميز.
- ❌ *«إذًا الترميز بالهدف آمن.»* لا. فالتسريب لا يساوي شيئًا **هنا**، والسبب حسابي: فبخمس مدن يُحسب
  متوسط كل مجموعة من نحو ألف صف ولا يكاد يعرف شيئًا عن أي صف بعينه، وبأربعة آلاف مدينة يُحسب كل
  متوسط من صف أو صفين فيكون الهدف نفسه مكتوبًا في الخاصية. الشيفرة نفسها والفرق كارثي. فالضرر
  يتناسب مع **عدد الفئات**.
- ✅ *«في هذا العمود لا يهمّ اختيار الترميز، وأنا أعرف السبب.»* وهذا يخبرك أيضًا أين تصرف انتباهك بدلًا
  من ذلك: في الفقدان الذي ساوى ٠٫٠٠٩، وفي التسريب الذي ساوى ٠٫٢١.

</div>

## Section 3 — Stretch: write the report  (≈30 min)

Open-ended. Lower expectation of completeness — get through the first part, and treat the second as
homework if you run out of time.

Write `imputation_report.md`. It needs three sections and it is short:

1. **What was wrong** — the five defects, one line each, with the numbers you measured.
2. **The three strategies and their scores** — all three, in a table, including the ones you rejected.
   State that the leaking column was removed first, and what it had been worth.
3. **Why you chose what you chose** — and this section has to name the **cause** of the missingness,
   not the score. "The indicator won by 0.003" is not a reason. "Phone orders never had the field, so
   the absence marks a channel rather than a discount of zero" is.

Then answer the question that separates an engineer from someone following a recipe:

> **If the three scores had been identical to four decimals, how would you have chosen?**

They nearly were. Write down what you would do, and be concrete: which strategy, and what evidence
would you have reached for instead of the score. Then say what it would cost if you were wrong —
imputing zero where the cause was "not recorded" biases every phone order in the same direction, and
a bias that has a direction is worse than noise.

**Link to your capstone:** your report is graded on the rationale for each preprocessing decision,
not on the final number. This section is that, in miniature.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: اكتب التقرير (نحو ٣٠ دقيقة)

قسم مفتوح، ولا يُتوقّع إكماله بالكامل — أنجز الجزء الأول واعتبر الثاني واجبًا منزليًا إن ضاق الوقت.

اكتب ملف `imputation_report.md`. ويحتاج ثلاثة أقسام، وهو قصير:

١. **ما كان معطوبًا** — العيوب الخمسة، سطر لكل واحد، مع الأرقام التي قِستها.
٢. **الطرائق الثلاث ونتائجها** — الثلاث كلها في جدول، ومنها ما رفضته. وصرّح بأن العمود المُسرِّب حُذف
   أولًا، وبما كان يساويه.
٣. **لماذا اخترت ما اخترت** — وعلى هذا القسم أن يذكر **سبب** الفقدان لا النتيجة. فقولك «فاز المؤشّر
   بـ ٠٫٠٠٣» ليس سببًا، وقولك «طلبات الهاتف لم يكن فيها هذا الحقل، فالغياب يدلّ على قناة لا على خصم
   قيمته صفر» سبب.

ثم أجب عن السؤال الذي يفصل المهندس عمّن يتبع وصفة:

> **لو كانت النتائج الثلاث متطابقة إلى أربع منازل، فكيف كنت ستختار؟**

وقد كانت تكاد تكون كذلك. فاكتب ما ستفعله، وكن محدّدًا: أي طريقة، وأي دليل كنت ستطلبه بدل النتيجة. ثم
قل ما يكلّفه الخطأ: فالتعويض بصفر حيث كان السبب «غير مُسجَّل» يُحيز كل طلبات الهاتف في اتجاه واحد،
والانحياز الذي له اتجاه أسوأ من الضوضاء.

**الصلة بمشروعك:** يُقيَّم تقريرك على مبرّر كل قرار معالجة لا على الرقم النهائي. وهذا القسم هو ذلك مصغَّرًا.

</div>

In [ ]:

# TODO: Write imputation_report.md with three sections, using the numbers you measured.
# مهمة: اكتب ملف imputation_report.md بثلاثة أقسام، مستخدمًا الأرقام التي قِستها.

**Your answer to the harder question:** _(if the three scores had been identical, how would you have
chosen, and what would being wrong cost?)_

<div dir="rtl" align="right">

**إجابتك عن السؤال الأصعب:** _(لو تطابقت النتائج الثلاث، كيف كنت ستختار، وما ثمن الخطأ؟)_

</div>

## Save your artefact

`cleaned.parquet` is tomorrow's input. It is the deduplicated, spelling-normalised, date-parsed,
imputed frame **with the leaking column gone** — which is the whole point of saving it rather than
re-cleaning tomorrow: the decisions travel with the data.

Save the winning imputation variant, not the raw frame.

<div dir="rtl" align="right">

## احفظ مخرجاتك

الملف `cleaned.parquet` هو مُدخل الغد. وهو الجدول بعد حذف المكرّر وتوحيد التهجئة وتحليل التواريخ
والتعويض، و**بلا العمود المُسرِّب** — وهذا هو سبب حفظه بدل إعادة التنظيف غدًا: فالقرارات تُسافر مع
البيانات.

احفظ نسخة التعويض الفائزة لا الجدول الخام.

</div>

In [ ]:
cleaned = by_indicator

out = ARTEFACT_DIR / "cleaned.parquet"
cleaned.to_parquet(out, index=False)

reloaded = pd.read_parquet(out)
print(f"Saved {out}")
print(f"{reloaded.shape[0]:,} rows x {reloaded.shape[1]} columns")
print(f"columns: {list(reloaded.columns)}")
print(f"round-trips to an equal DataFrame: {reloaded.equals(cleaned)}")
print(f"missing values remaining: {reloaded.isna().sum().sum()}")

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخلية أخيرًا. كل فحص يفشل يخبرك بما يجب إصلاحه ولماذا.

</div>

In [ ]:
# --- Sanity checks ----------------------------------------------------------------

check(cleaned.duplicated().sum() == 0,
      f"the cleaned frame must have no duplicate rows, found {cleaned.duplicated().sum()}",
      f"يجب ألا يحتوي الجدول المُنظَّف صفوفًا مكرّرة، والموجود {cleaned.duplicated().sum()}")

check(cleaned["city"].nunique() == 5,
      f"city should have exactly 5 distinct values after normalising, got "
      f"{cleaned['city'].nunique()}: {sorted(cleaned['city'].unique())}",
      f"يجب أن يحتوي عمود المدينة خمس قيم مختلفة بعد التوحيد، والموجود "
      f"{cleaned['city'].nunique()}: {sorted(cleaned['city'].unique())}")

check(n_unparsed == 0 and n_future == 0 and n_before_order == 0,
      f"dates must all parse, none in the future, none shipped before ordered "
      f"(unparsed {n_unparsed}, future {n_future}, shipped-early {n_before_order})",
      f"يجب أن تُحلَّل كل التواريخ، ولا تاريخ في المستقبل، ولا شحن قبل الطلب "
      f"(فاشل {n_unparsed}، مستقبلي {n_future}، شُحن مبكرًا {n_before_order})")

check(len(by_median) == len(by_zero) == len(by_indicator) == len(clean),
      f"the three imputation variants must all have the same row count "
      f"({len(by_median):,} / {len(by_zero):,} / {len(by_indicator):,})",
      f"يجب أن تتساوى نسخ التعويض الثلاث في عدد الصفوف "
      f"({len(by_median):,} / {len(by_zero):,} / {len(by_indicator):,})")

check(LEAK not in cleaned.columns,
      f"the leaking column {LEAK!r} must not be in the frame you saved — "
      f"columns are {list(cleaned.columns)}",
      f"يجب ألا يكون العمود المُسرِّب {LEAK!r} في الجدول الذي حفظته — "
      f"والأعمدة هي {list(cleaned.columns)}")

check(first_r2 - honest_r2 > 0.10,
      f"dropping the leak should cost you more than 0.10 of R2 — with it {first_r2:.4f}, "
      f"without it {honest_r2:.4f}. If the gap is small you have not actually removed it.",
      f"يجب أن يكلّفك حذف التسريب أكثر من ٠٫١٠ من R² — معه {first_r2:.4f} وبدونه "
      f"{honest_r2:.4f}. فإن كان الفرق صغيرًا فأنت لم تحذفه فعلًا.")

sections = [line for line in (ARTEFACT_DIR / "imputation_report.md").read_text(
    encoding="utf-8").splitlines() if line.startswith("## ")]
check(len(sections) >= 3,
      f"imputation_report.md needs at least 3 sections, found {len(sections)}: {sections}",
      f"يحتاج ملف imputation_report.md ثلاثة أقسام على الأقل، والموجود {len(sections)}: {sections}")

report()

## What's next

Tomorrow (**W2D4**) you load this `cleaned.parquet` and do the opposite of today: instead of removing
damage you add signal. Five new features, measured **one at a time**, so that at the end you can say
which idea worked rather than that the batch did. Some of them will make the model worse, and those
rows stay in the table.

One of the five will recover 0.19 of the 0.21 of R² that dropping the commission cost you —
legitimately, from columns that exist the moment an order is placed. That is the honest route to
information a leaking column was handing you for free, and it is worth seeing the two side by side.

<div dir="rtl" align="right">

## ماذا بعد

غدًا (**الأسبوع ٢ اليوم ٤**) تُحمّل هذا الملف `cleaned.parquet` وتفعل نقيض اليوم: فبدل إزالة الضرر
تضيف إشارة. خمس خصائص جديدة، تُقاس **واحدة واحدة**، حتى تستطيع في النهاية أن تقول أي فكرة نجحت لا أن
المجموعة نجحت. وبعضها سيجعل النموذج أسوأ، وتبقى تلك الصفوف في الجدول.

وستستعيد إحدى الخمس ٠٫١٩ من الـ ٠٫٢١ التي كلّفك حذف العمولة — استعادةً مشروعة من أعمدة موجودة لحظة
تقديم الطلب. وهذا هو الطريق الصادق إلى معلومة كان العمود المُسرِّب يعطيك إياها مجانًا، ويستحق الأمر أن
تراهما جنبًا إلى جنب.

</div>